# 04 — Enrichissement programme et variables métier

Cette étape extrait la famille de cursus, l'année, la majeure et le campus explicitement déclaré dans le libellé. Elle sépare strictement trois concepts : programme SWIM, type de stage SWIM et stage international.

Sortie **intermédiaire** : `05_normalisation_entreprises.ipynb` consomme ce fichier et produit le dataset final unique (`data/processed/stages_efrei_final.csv`).

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from stages_cleaning.programme import (
    assign_major_canonical,
    enrich_programme_features,
    major_canonicalization_issues,
    programme_validation_issues,
)

INPUT_PATH = ROOT / "data" / "interim" / "stages_efrei_with_postal_code.csv"
OUTPUT_PATH = ROOT / "data" / "interim" / "stages_efrei_with_programme.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_PATH, dtype="string", encoding="utf-8-sig", keep_default_na=False).replace("", pd.NA)
print("Shape entrée :", df.shape)

## 1. Extraction des features

In [ ]:
final = enrich_programme_features(df)
added = [c for c in final.columns if c not in df.columns]
print("Colonnes ajoutées :", added)
print(final[["real_program", "program_year", "major_status", "programme_campus"]].describe(include="all"))

## 2bis. Canonisation des majeures (référentiel officiel EFREI)

Rattache chaque `major` à son nom officiel actuel (audit manuel du site EFREI, voir `programme.py::MAJOR_CANONICAL`), à sa filière/famille (`programme_groupe`), et à un statut de confiance explicite (`major_canonical_status`) :
- `official` : nom officiel confirmé sur le site.
- `kept_as_is` : majeure discontinuée, conservée telle quelle (ex. `Digital Transformation`).
- `pending_review` : programme réel et actif mais absent du menu actuel du site — volontairement non renommé pour éviter une fusion silencieuse incorrecte.

Aucune valeur de `major` n'est jamais vidée par cette étape.

In [ ]:
final = assign_major_canonical(final)

print(final["major_canonical_status"].value_counts(dropna=False))
print()
print("Filière/famille (programme_groupe) :")
print(final["programme_groupe"].value_counts(dropna=False))

unknown = major_canonicalization_issues(final)
assert unknown.empty, (
    f"{len(unknown)} valeur(s) de major_canonical hors référentiel — "
    "mettre à jour MAJOR_CANONICAL / MAJOR_KEEP_AS_IS / MAJOR_PENDING_REVIEW dans programme.py"
)
print("\n✔ Toute valeur de major_canonical est couverte par le référentiel (officielle, conservée, ou explicitement en attente).")

pending = final.loc[final["major_canonical_status"].eq("pending_review"), "major_canonical"]
if not pending.empty:
    print(f"\n⚠ {len(pending)} ligne(s) en attente de nom officiel (programmes réels, "
          "non fusionnés par sécurité) :")
    print(pending.value_counts())


## 2. Contrôles de parsing

In [ ]:
issues = programme_validation_issues(final)
for name, frame in issues.items():
    print(name, ":", len(frame))
    if not frame.empty:
        display(frame[["stage_id", "programme", "real_program", "program_year", "major", "major_status"]].head(30))
# Option (b) retenue (AUDIT_FIXES.md C-5) : on signale sans bloquer la livraison, cohérent avec
# la philosophie "détecter, pas bloquer" du registre qualité. La règle PROGRAMME_UNRECOGNIZED
# (voir quality_reporting/rules.py) prend le relais dans le rapport pour le suivi humain.
if not all(frame.empty for frame in issues.values()):
    print("⚠ Le parsing programme contient des anomalies (voir détail ci-dessus) ; "
          "elles remontent dans le rapport qualité via la règle PROGRAMME_UNRECOGNIZED, "
          "la livraison n'est pas bloquée.")
else:
    print("✔ Aucune anomalie de parsing programme")

## 3. Validation des trois notions distinctes

In [ ]:
print("Programme [SWIM] :", int(final["is_swim_program"].sum()))
print("Type stage SWIM :", int(final["is_swim_stage"].sum()))
print("Stage international connu :", int(final["is_international_stage"].sum(skipna=True)))
print("Pays inconnu → international inconnu :", int(final["is_international_stage"].isna().sum()))
assert final.loc[final["pays"].isna(), "is_international_stage"].isna().all()
assert final["is_swim_stage"].equals(final["type_stage"].str.strip().str.upper().eq("SWIM").astype("boolean"))
display(pd.crosstab(final["is_swim_program"], final["is_swim_stage"], margins=True))

## 4. Export final

In [ ]:
assert len(final) == len(df)
assert final["stage_id"].equals(df["stage_id"])
final.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print("Export intermédiaire (programme) :", OUTPUT_PATH)
print("Shape :", final.shape)